# 00. Dataset Construction and Storage — GSE69914

This notebook builds the core methylation matrix used throughout the thesis.  
It documents the full ingestion pipeline — from raw GEO Series Matrix parsing to transposed, labeled, and compressed Parquet storage — ensuring reproducibility, memory efficiency, and schema consistency across downstream analyses.

**Source: GEO accession GSE69914, platform Illumina HumanMethylation450 BeadChip**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-preparation-GSE69914                    ║
# ║ Description:  Construction of the core methylation matrix        ║
# ║               (GSE69914) — parsing, transposition, labeling,     ║
# ║               and Parquet storage.                               ║
# ║ Dataset(s):   GSE69914                                           ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 01-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [ ]:
!pip -q install GEOparse polars pyarrow


In [ ]:
import GEOparse
import pandas as pd
import polars as pl
import os, re, csv
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import numpy as np


## 1. Fetch numeric labels from GEO (0..4)

In [ ]:
# FETCH NUMERIC LABELS FROM GEO
LABELS_CSV = "/kaggle/working/GSE69914_labels_numeric.csv"

# Download + parse GEO entry (lightweight vs matrix)
gse = GEOparse.get_GEO("GSE69914", destdir=".")

rows = []
for gsm_name, gsm in gse.gsms.items():
    txt = " | ".join([f"{k}: {v}" for k, v in gsm.metadata.items()]).lower()
    m = re.search(r"status\s*\([^)]*\)\s*:\s*([0-4])", txt)  # status(...): X
    code = int(m.group(1)) if m else None
    rows.append({"gsm": gsm_name, "status_code": code})

labels_df = pd.DataFrame(rows, columns=["gsm", "status_code"])
labels_df.to_csv(LABELS_CSV, index=False)

print(f"✅ Labels saved: {LABELS_CSV}")
print(labels_df["status_code"].value_counts(dropna=False).sort_index())


## 2. Read TXT → transpose to Sample×CpG → attach labels → write Parquet (LZ4)

In [ ]:
# TXT -> TRANSPOSE -> ATTACH LABELS -> PARQUET LZ4
INPUT_TXT  = "/kaggle/input/gse69914-series-matrix-txt/GSE69914_series_matrix.txt"  # change if needed
LABELS_CSV = "/kaggle/working/GSE69914_labels_numeric.csv"
OUT_PAR    = "/kaggle/working/GSE69914_beta_with_labels_numeric_lz4.parquet"

assert os.path.exists(INPUT_TXT),  f"Missing: {INPUT_TXT}"
assert os.path.exists(LABELS_CSV), f"Missing: {LABELS_CSV} (run Cell 1 first)"

# 1) Load labels into a pandas Series for fast mapping
lab = pd.read_csv(LABELS_CSV)
lab = lab.dropna(subset=["gsm"]).copy()
lab["gsm"] = lab["gsm"].astype(str)
lab_series = pd.Series(lab["status_code"].astype("Int8").values, index=lab["gsm"].values)

# 2) Read the GEO Series Matrix (tab-delimited), skip header metadata
#    We load as float then downcast to float32 to save RAM.
df = pd.read_csv(
    INPUT_TXT,
    sep="\t",
    skiprows=73,      # skip the 73-line GEO header
    comment="!",      # ignore trailing metadata lines
    engine="c",
    dtype=None,       # let pandas infer, we'll cast after
    low_memory=False,
)

# Ensure first column is ID_REF (CpG / probe ID)
if df.columns[0] != "ID_REF":
    df.rename(columns={df.columns[0]: "ID_REF"}, inplace=True)

# 3) Cast all sample columns to float32 (keep ID_REF as string)
sample_cols = df.columns.tolist()[1:]
for c in sample_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

# 4) Transpose to Sample x CpG (rows=samples, cols=probes)
df_T = df.set_index("ID_REF").T
df_T.index.name = "id_tissue"   # index are GSM sample IDs

# 5) Bring index to a column, attach numeric label, enforce compact dtypes
df_T = df_T.reset_index()                 # id_tissue becomes a column
df_T.insert(1, "label", lab_series.reindex(df_T["id_tissue"]).values)  # insert as 2nd column
df_T["label"] = df_T["label"].astype("Int8")

# 6) Convert to Arrow Table with explicit schema and write Parquet (LZ4)
#    Build fields: id_tissue=utf8, label=int8, all probe columns=float32
fields = [pa.field("id_tissue", pa.string()),
          pa.field("label", pa.int8())]
for col in df_T.columns[2:]:
    fields.append(pa.field(col, pa.float32()))
schema = pa.schema(fields)

# Create Arrow Table without copying data when possible
table = pa.Table.from_pandas(df_T, preserve_index=False, schema=schema)

# Write Parquet with LZ4 (fast & lossless). Dictionary encoding helps metadata.
pq.write_table(
    table,
    OUT_PAR,
    compression="lz4",
    use_dictionary=True
)

print(f"✅ Parquet saved: {OUT_PAR}")

# 7) Quick sanity checks: shape and a tiny peek (fast)
par = pl.read_parquet(OUT_PAR)          # full read (ensure it loads)
print("Parquet shape:", par.shape)      # expected ~ (407, ~485k + 2)
print(par.select(["id_tissue","label"]).head())


## 3. Ultra-Fast Future Reading (Polars)

In [ ]:
# ULTRA-FAST FUTURE READING
# 1) schema/shape via metadata (zero-cost-ish)
meta = pq.ParquetFile(OUT_PAR).metadata
print("Rows:", meta.num_rows, "Cols:", meta.num_columns)

# 2) load full once (you said you'll always load all)
df = pl.read_parquet(OUT_PAR)
print(df.shape)                 # ~ (407, ~485k+2)
assert "id_tissue" in df.columns and "label" in df.columns
assert df["label"].cast(pl.Int64).is_in([0,1,2,3,4]).all(), "Unexpected label codes"
print(df.head())
